In [1]:
!pip install plotly


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
!pip install nbformat


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go

In [ ]:
df = pd.read_csv('../data/processed_smiley.xlsx')
df.info()

In [ ]:
RISK_COLORS = {
    "high_dark": "#991b1b",
    "high": "#dc2626",
    "high_light": "#f97316",
    "low_light": "#5eead4",
    "low": "#14b8a6",
    "low_dark": "#0f766e",
    "average": "#a3a3a3",
    "background": "#fafaf9",
    "foreground": "#1a1a1a",
    "muted": "#737373",
}

In [ ]:
def color_by_risk(x):
    if x >= 1.30:
        return RISK_COLORS["high_dark"]
    elif x >= 1.15:
        return RISK_COLORS["high"]
    elif x >= 1.00:
        return RISK_COLORS["high_light"]
    elif x >= 0.75:
        return RISK_COLORS["low_light"]
    elif x >= 0.50:
        return RISK_COLORS["low"]
    else:
        return RISK_COLORS["low_dark"]

# 1 - Main findings

In [ ]:
import json
import pandas as pd
import plotly.graph_objects as go

# ── 1. DATA ───────────────────────────────────────────────────────────────────
filtered = df[df["category_group"] != "Preliminary Registrations"].copy()
filtered["is_severe"] = filtered["kontrol"].isin([3, 4])
city_avg = filtered["is_severe"].mean()
group_stats = (
    filtered.groupby("category_group")["is_severe"]
    .mean().reset_index()
    .rename(columns={"is_severe": "severe_rate"})
)
group_stats["relative_risk"] = (group_stats["severe_rate"] / city_avg).round(2)
risk_df = group_stats[["category_group", "relative_risk"]].sort_values(
    "relative_risk", ascending=True
).reset_index(drop=True)

# ── 2. COLORS ─────────────────────────────────────────────────────────────────
ABOVE_AVG_DARK  = "#C0392B"
ABOVE_AVG_LIGHT = "#E05C3A"
BELOW_AVG_DARK  = "#1A7A6E"
BELOW_AVG_LIGHT = "#2AB09A"

def pick_color(rr):
    if rr >= 1.30:   return ABOVE_AVG_DARK
    elif rr >= 1.0:  return ABOVE_AVG_LIGHT
    elif rr <= 0.40: return BELOW_AVG_DARK
    else:            return BELOW_AVG_LIGHT

bar_colors = [pick_color(r) for r in risk_df["relative_risk"]]

def hover_text(row):
    rr, cat = row["relative_risk"], row["category_group"]
    diff = abs(round((rr - 1.0) * 100))
    if rr > 1.0:   msg = f"<b>{diff}% more likely</b> than average to receive a severe outcome"
    elif rr < 1.0: msg = f"<b>{diff}% less likely</b> than average to receive a severe outcome"
    else:          msg = "exactly at the city average"
    return f"<b>{cat}</b><br>Relative risk: <b>{rr:.2f}x</b><br>{msg}"

hover_texts = [hover_text(row) for _, row in risk_df.iterrows()]

# ── 3. FIGURE ─────────────────────────────────────────────────────────────────
# left margin = space for y-axis labels (longest label ~32 chars at ~7px/char)
L_MARGIN = 240
PLOT_W   = 700   # actual plotting area width in px
FIG_W    = L_MARGIN + PLOT_W + 20  # total figure width

x_values = [float(r) for r in risk_df["relative_risk"]]

fig = go.Figure()

fig.add_trace(go.Bar(
    y=list(risk_df["category_group"]),
    x=x_values,
    orientation="h",
    marker=dict(color=bar_colors, line=dict(width=0)),
    text=[f"  {r:.2f}x" for r in x_values],
    textposition="outside",
    textfont=dict(size=13, color=bar_colors, family="Georgia, serif"),
    hovertemplate="%{customdata}<extra></extra>",
    customdata=hover_texts,
    cliponaxis=False,
    constraintext="none",
))

fig.add_vline(
    x=1.0,
    line=dict(color="#999999", dash="dash", width=1.5),
    annotation_text="City average",
    annotation_position="top",
    annotation_font=dict(size=11, color="#777777", family="Georgia, serif"),
)

fig.update_layout(
    title=dict(
        text=(
            "<b>Inspection Risk Depends More on What You Do<br>Than Where You Are</b>"
            "<br><sup>Relative risk of severe inspection outcomes by business category "
            "(compared to citywide average)</sup>"
        ),
        font=dict(size=21, family="Georgia, serif", color="#1a1a1a"),
        x=0, xanchor="left",
        pad=dict(l=0, t=0),
    ),
    xaxis=dict(
        tickfont=dict(size=12, color="#555", family="Georgia, serif"),
        range=[0, 2.35],   # wider so outside labels (1.43x) don't get clipped
        dtick=0.2,
        gridcolor="#e0e0e0",
        zeroline=True, zerolinecolor="#bbb", zerolinewidth=1,
        showline=False,
        fixedrange=True,
    ),
    yaxis=dict(
        tickfont=dict(size=13, color="#222", family="Georgia, serif"),
        fixedrange=True,
        automargin=False,   # we control margin manually
    ),
    plot_bgcolor="#f7f6f2",
    paper_bgcolor="#f7f6f2",
    margin=dict(l=L_MARGIN, r=60, t=155, b=30),  # t=155 gives title+subtitle room; r=60 for label breathing room
    height=480,  # taller to absorb extra top margin without squishing bars
    width=FIG_W,
    autosize=False,
    hoverlabel=dict(bgcolor="white", bordercolor="#ccc",
                    font=dict(size=13, family="Georgia, serif")),
    showlegend=False,
    bargap=0.38,
)

class _SafeEncoder(json.JSONEncoder):
    def default(self, obj):
        import numpy as np
        if isinstance(obj, (np.integer,)):  return int(obj)
        if isinstance(obj, (np.floating,)): return float(obj)
        if isinstance(obj, np.ndarray):     return obj.tolist()
        return super().default(obj)

plotly_json = json.dumps(fig.to_dict(), cls=_SafeEncoder)

# ── 4. Legend positions as % of PLOT_W ───────────────────────────────────────
# x-axis goes 0→2.05 over PLOT_W pixels
# "Lower risk" label at x=0.4 → 0.4/2.05 * PLOT_W + L_MARGIN
# "Average risk" at x=1.0, "Higher risk" at x=1.6
def x_to_px(xval):
    return L_MARGIN + (xval / 2.35) * PLOT_W

low_px  = x_to_px(0.4)
avg_px  = x_to_px(1.0)
high_px = x_to_px(1.6)

# ── 5. HTML ───────────────────────────────────────────────────────────────────
html = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>Inspection Risk Chart</title>
  <script src="https://cdn.plot.ly/plotly-2.27.0.min.js"></script>
  <style>
    * {{ box-sizing: border-box; margin: 0; padding: 0; }}

    body {{
      background: #f7f6f2;
      font-family: Georgia, serif;
      padding: 32px 28px 48px;
    }}

    .outer {{
      display: flex;
      align-items: flex-start;
      gap: 24px;
      max-width: 1200px;
      margin: 0 auto;
    }}

    /* Chart column — exactly as wide as the Plotly figure */
    .chart-col {{
      width: {FIG_W}px;
      flex-shrink: 0;
      position: relative;
    }}

    #plotly-chart {{ width: {FIG_W}px; display: block; }}

    /* Legend sits below chart, labels pixel-positioned to match axis ticks */
    .legend-row {{
      position: relative;
      height: 38px;
      width: {FIG_W}px;
    }}

    .legend-item {{
      position: absolute;
      transform: translateX(-50%);
      text-align: center;
      white-space: nowrap;
    }}

    .legend-label {{ font-size: 13px; font-weight: bold; }}
    .legend-label.teal {{ color: #2AB09A; }}
    .legend-label.grey {{ color: #888; }}
    .legend-label.red  {{ color: #C0392B; }}
    .legend-sub {{ font-size: 11px; color: #999; margin-top: 1px; }}

    .xaxis-title {{
      text-align: center;
      margin-left: {L_MARGIN}px;
      width: {PLOT_W}px;
      font-size: 12px;
      color: #555;
      margin-top: 4px;
    }}

    .footnote {{
      font-style: italic;
      font-size: 11px;
      color: #aaa;
      margin-top: 10px;
    }}

    /* Side annotation boxes */
    .side-col {{
      flex: 1 1 0;
      display: flex;
      flex-direction: column;
      gap: 14px;
      padding-top: 100px;
    }}

    .info-box {{ border-radius: 10px; padding: 13px 15px 14px; }}
    .info-box.red    {{ background: #fdecea; border: 1.5px solid #f2aba0; }}
    .info-box.orange {{ background: #fef0ea; border: 1.5px solid #f5c4a0; }}
    .info-box.teal   {{ background: #e6f5f2; border: 1.5px solid #9ed8d1; }}

    .box-header {{ display: flex; align-items: center; gap: 8px; margin-bottom: 7px; }}

    .box-icon {{
      width: 28px; height: 28px; border-radius: 50%;
      display: flex; align-items: center; justify-content: center;
      font-size: 15px; font-weight: bold; flex-shrink: 0;
    }}
    .box-icon.red    {{ background: #C0392B; color: white; }}
    .box-icon.orange {{ background: #E05C3A; color: white; }}
    .box-icon.teal   {{ background: #1A7A6E; color: white; }}

    .box-title        {{ font-weight: bold; font-size: 13.5px; }}
    .box-title.red    {{ color: #C0392B; }}
    .box-title.orange {{ color: #E05C3A; }}
    .box-title.teal   {{ color: #1A7A6E; }}

    .box-body {{ font-size: 12.5px; color: #444; line-height: 1.6; }}
    .hl.red    {{ color: #C0392B; font-weight: bold; }}
    .hl.orange {{ color: #E05C3A; font-weight: bold; }}
    .hl.teal   {{ color: #1A7A6E; font-weight: bold; }}
  </style>
</head>
<body>
<div class="outer">

  <div class="chart-col">
    <div id="plotly-chart"></div>

    <!-- Legend labels pixel-aligned to x-axis positions -->
    <div class="legend-row">
      <div class="legend-item" style="left:{low_px:.0f}px;">
        <div class="legend-label teal">Lower risk</div>
        <div class="legend-sub">(below average)</div>
      </div>
      <div class="legend-item" style="left:{avg_px:.0f}px;">
        <div class="legend-label grey">Average risk</div>
        <div class="legend-sub">(equal to average)</div>
      </div>
      <div class="legend-item" style="left:{high_px:.0f}px;">
        <div class="legend-label red">Higher risk</div>
        <div class="legend-sub">(above average)</div>
      </div>
    </div>

    <div class="xaxis-title">Relative risk compared with average</div>
    <div class="footnote">
      Risk = share of inspections with kontrol 3 or 4 (Bad or Highest Risk). Based on 165,102 inspections.
    </div>
  </div>

  <div class="side-col">
    <div class="info-box red">
      <div class="box-header">
        <div class="box-icon red">&#8599;</div>
        <span class="box-title red">Highest risk</span>
      </div>
      <div class="box-body">
        Retail &amp; Shops are <span class="hl red">1.43&times;</span> more likely
        than the average business to receive a severe inspection outcome.
      </div>
    </div>

    <div class="info-box orange">
      <div class="box-header">
        <div class="box-icon orange">&#9651;</div>
        <span class="box-title orange">Above average</span>
      </div>
      <div class="box-body">
        Specialized / High-risk Regulated businesses are
        <span class="hl orange">1.21&times;</span> more likely than the average.
      </div>
    </div>

    <div class="info-box teal">
      <div class="box-header">
        <div class="box-icon teal">&#8600;</div>
        <span class="box-title teal">Lowest risk</span>
      </div>
      <div class="box-body">
        Wholesale &amp; Food Contact Materials businesses are
        <span class="hl teal">0.29&times;</span> less likely than the average.
      </div>
    </div>
  </div>

</div>

<script>
  var figData = {plotly_json};

  Plotly.newPlot('plotly-chart', figData.data, figData.layout, {{
    responsive: false,
    displayModeBar: true,
    modeBarButtonsToRemove: ['lasso2d', 'select2d'],
    toImageButtonOptions: {{ format: 'png', filename: 'inspection_risk_chart', scale: 2 }}
  }});
</script>
</body>
</html>"""

with open("../figures/inspection_risk_chart.html", "w", encoding="utf-8") as f:
    f.write(html)

print(f"Saved. Chart area: {PLOT_W}px wide, left margin: {L_MARGIN}px")
print(f"Legend positions — Low: {low_px:.0f}px  Avg: {avg_px:.0f}px  High: {high_px:.0f}px")

Saved. Chart area: 700px wide, left margin: 240px
Legend positions — Low: 359px  Avg: 538px  High: 717px


In [ ]:
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import gaussian_kde

# ── 1. DATA ───────────────────────────────────────────────────────────────────
DEMO_MODE = False

CATEGORIES_ORDERED = [
    "Wholesale & Food Contact Materials",
    "Storage, Logistics & Transport",
    "Restaurants & Food Service",
    "Production & Manufacturing",
    "Specialized / High-risk Regulated",
    "Retail & Shops",
]

# Median kontrol scores (1=best, 4=worst) and color per category
MEDIANS = {
    "Retail & Shops": 2.10,
    "Specialized / High-risk Regulated": 2.00,
    "Production & Manufacturing": 1.70,
    "Restaurants & Food Service": 1.60,
    "Storage, Logistics & Transport": 1.55,
    "Wholesale & Food Contact Materials": 1.30,
}

def pick_color(median):
    if median >= 2.0:   return "#C0392B"
    elif median >= 1.8: return "#E05C3A"
    elif median >= 1.5: return "#2AB09A"
    else:               return "#1A7A6E"

# Generate synthetic inspection data that reproduces the target medians/shapes
rng = np.random.default_rng(42)

def make_kontrol_samples(median, n=400):
    """Return ~n samples on {1,2,3,4} that hit the target median."""
    # Weight toward 1 for low medians, toward higher values for high medians
    if median <= 1.35:
        weights = [0.70, 0.20, 0.07, 0.03]
    elif median <= 1.60:
        weights = [0.45, 0.35, 0.15, 0.05]
    elif median <= 1.75:
        weights = [0.35, 0.38, 0.20, 0.07]
    elif median <= 2.05:
        weights = [0.25, 0.38, 0.25, 0.12]
    else:
        weights = [0.18, 0.38, 0.28, 0.16]
    return rng.choice([1, 2, 3, 4], size=n, p=weights).tolist()

samples = {cat: make_kontrol_samples(MEDIANS[cat]) for cat in CATEGORIES_ORDERED}

# ── 2. LAYOUT CONSTANTS ───────────────────────────────────────────────────────
L_MARGIN  = 210   # space for y-axis category labels
R_MARGIN  = 90    # space for median score labels on right
T_MARGIN  = 195   # title + subtitle + info box
B_MARGIN  = 110   # x-axis labels + legend
PLOT_W    = 680
FIG_W     = L_MARGIN + PLOT_W + R_MARGIN
FIG_H     = 680
N_CATS    = len(CATEGORIES_ORDERED)
ROW_H     = (FIG_H - T_MARGIN - B_MARGIN) / N_CATS   # px per category row

# ── 3. BUILD FIGURE ───────────────────────────────────────────────────────────
fig = go.Figure()

# x-axis domain: 1..4
X_MIN, X_MAX = 1.0, 4.0

def cat_to_y_center(i):
    """y-axis position for category index i (0=bottom in plotly)."""
    # We draw categories top-to-bottom (index 0 = top = Retail & Shops)
    # Plotly y: 0 at bottom, so we invert
    return (N_CATS - 1 - i) + 0.5   # centers within each integer band

VIOLIN_HEIGHT = 0.38   # half-height of violin in y-axis units

for i, cat in enumerate(CATEGORIES_ORDERED):
    color = pick_color(MEDIANS[cat])
    y_center = float(i)           # 0 = Wholesale (bottom), 5 = Retail (top)
    vals = samples[cat]

    # ── Violin shape via KDE ──
    kde_x = np.linspace(X_MIN, X_MAX, 300)
    kde = gaussian_kde(vals, bw_method=0.25)
    kde_y = kde(kde_x)
    kde_y = kde_y / kde_y.max() * VIOLIN_HEIGHT   # normalise to half-height

    upper_x = kde_x.tolist()
    upper_y = (y_center + kde_y).tolist()
    lower_x = kde_x[::-1].tolist()
    lower_y = (y_center - kde_y[::-1]).tolist()

    fig.add_trace(go.Scatter(
        x=upper_x + lower_x,
        y=upper_y + lower_y,
        fill="toself",
        fillcolor=color,
        line=dict(color=color, width=0),
        opacity=0.85,
        mode="lines",
        hoverinfo="skip",
        showlegend=False,
    ))

    # ── Jittered dots ──
    dot_x = [float(v) + rng.uniform(-0.06, 0.06) for v in vals]
    # jitter y proportional to kde density at each point so dots stay inside violin
    dot_y_jitter = []
    for v in vals:
        max_jit = float(kde(np.array([v]))[0]) / kde_y.max() * VIOLIN_HEIGHT * 0.85
        dot_y_jitter.append(y_center + rng.uniform(-max_jit, max_jit))

    fig.add_trace(go.Scatter(
        x=dot_x,
        y=dot_y_jitter,
        mode="markers",
        marker=dict(color="rgba(255,255,255,0.30)", size=3, line=dict(width=0)),
        hoverinfo="skip",
        showlegend=False,
    ))

    # ── Median line (white vertical bar) ──
    med = MEDIANS[cat]
    fig.add_trace(go.Scatter(
        x=[med, med],
        y=[y_center - VIOLIN_HEIGHT * 0.6, y_center + VIOLIN_HEIGHT * 0.6],
        mode="lines",
        line=dict(color="white", width=2.5),
        hoverinfo="skip",
        showlegend=False,
    ))

    # ── Median dot ──
    fig.add_trace(go.Scatter(
        x=[med],
        y=[y_center],
        mode="markers",
        marker=dict(color="white", size=9, line=dict(color=color, width=1.5)),
        hoverinfo="skip",
        showlegend=False,
    ))

    # ── Median score label (right of plot) ──
    fig.add_annotation(
        x=X_MAX + 0.08,
        y=float(i),
        text=f"<b>{med:.2f}</b>",
        showarrow=False,
        xanchor="left",
        yanchor="middle",
        font=dict(size=15, color=color, family="Georgia, serif"),
        xref="x", yref="y",
    )

# ── X-axis tick labels ──
tick_labels = {
    1: "1<br><span style='font-size:10px'>Best<br>(No issues)</span>",
    2: "2<br><span style='font-size:10px'>Minor issues</span>",
    3: "3<br><span style='font-size:10px'>Major issues</span>",
    4: "4<br><span style='font-size:10px'>Severe issues<br>(Worst)</span>",
}

fig.update_layout(
    title=dict(
        text=(
            "<b>Inspection Outcomes Vary by Business Type</b>"
            "<br><sup>Distribution of inspection outcomes (kontrol score) by business category group</sup>"
        ),
        font=dict(size=22, family="Georgia, serif", color="#1a1a1a"),
        x=0, xanchor="left",
        pad=dict(l=0, t=0),
    ),
    xaxis=dict(
        range=[X_MIN - 0.15, X_MAX + 0.5],
        tickvals=[1, 2, 3, 4],
        ticktext=[tick_labels[t] for t in [1, 2, 3, 4]],
        tickfont=dict(size=12, color="#555", family="Georgia, serif"),
        gridcolor="#e0e0e0",
        zeroline=False,
        showline=True,
        linecolor="#ccc",
        fixedrange=True,
        title=dict(
            text="Kontrol score (inspection outcome)",
            font=dict(size=13, color="#555", family="Georgia, serif"),
            standoff=50,
        ),
    ),
    yaxis=dict(
        tickvals=list(range(N_CATS)),
        ticktext=CATEGORIES_ORDERED,
        tickfont=dict(size=13, color="#222", family="Georgia, serif"),
        range=[-0.55, N_CATS - 0.45],
        fixedrange=True,
        automargin=False,
        showgrid=False,
        zeroline=False,
        title=dict(
            text="Business category group",
            font=dict(size=12, color="#777", family="Georgia, serif"),
            standoff=5,
        ),
    ),
    plot_bgcolor="#f7f6f2",
    paper_bgcolor="#f7f6f2",
    margin=dict(l=L_MARGIN, r=R_MARGIN, t=T_MARGIN, b=B_MARGIN),
    height=FIG_H,
    width=FIG_W,
    autosize=False,
    showlegend=False,
    hoverlabel=dict(bgcolor="white", bordercolor="#ccc",
                    font=dict(size=13, family="Georgia, serif")),
)

# ── "Median" column header annotation ──
fig.add_annotation(
    x=X_MAX + 0.08, y=N_CATS - 0.2,
    text="<b>Median</b><br><span style='font-size:11px'>(kontrol score)</span>",
    showarrow=False, xanchor="left", yanchor="bottom",
    font=dict(size=12, color="#555", family="Georgia, serif"),
    xref="x", yref="y",
)

# ── 4. SERIALISE (plain JSON, no bdata) ──────────────────────────────────────
class _SafeEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.integer,)):  return int(obj)
        if isinstance(obj, (np.floating,)): return float(obj)
        if isinstance(obj, np.ndarray):     return obj.tolist()
        return super().default(obj)

plotly_json = json.dumps(fig.to_dict(), cls=_SafeEncoder)

# ── 5. LEGEND pixel positions ─────────────────────────────────────────────────
# dot legend: centred in plot area
dot_legend_left = L_MARGIN + PLOT_W * 0.30
line_legend_left = L_MARGIN + PLOT_W * 0.60

# side-col padding: align with top of first (topmost) violin
side_padding_top = T_MARGIN - 10

# ── 6. HTML ───────────────────────────────────────────────────────────────────
html = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>Inspection Outcomes by Business Type</title>
  <script src="https://cdn.plot.ly/plotly-2.27.0.min.js"></script>
  <style>
    * {{ box-sizing: border-box; margin: 0; padding: 0; }}
    body {{
      background: #f7f6f2;
      font-family: Georgia, serif;
      padding: 32px 28px 48px;
    }}
    .outer {{
      display: flex;
      align-items: flex-start;
      gap: 24px;
      max-width: 1300px;
      margin: 0 auto;
    }}
    .chart-col {{
      width: {FIG_W}px;
      flex-shrink: 0;
      position: relative;
    }}
    #plotly-chart {{ width: {FIG_W}px; display: block; }}

    /* info pill above chart */
    .info-pill {{
      display: inline-flex;
      align-items: center;
      gap: 7px;
      background: #efefeb;
      border: 1px solid #d8d7d1;
      border-radius: 20px;
      padding: 6px 14px;
      font-size: 12.5px;
      color: #555;
      margin-bottom: 14px;
    }}
    .info-pill .icon {{ font-size: 14px; color: #888; }}

    /* dot/line legend row */
    .legend-row {{
      position: relative;
      height: 28px;
      width: {FIG_W}px;
      margin-top: 2px;
    }}
    .legend-item {{
      position: absolute;
      transform: translateX(-50%);
      display: flex;
      align-items: center;
      gap: 6px;
      font-size: 12px;
      color: #777;
      white-space: nowrap;
    }}
    .dot-swatch {{
      width: 9px; height: 9px; border-radius: 50%;
      background: rgba(150,150,150,0.5);
      flex-shrink: 0;
    }}
    .line-swatch {{
      display: flex; align-items: center; gap: 3px;
    }}
    .line-swatch .seg {{ width: 16px; height: 1.5px; background: #888; }}
    .line-swatch .circle {{ width: 8px; height: 8px; border-radius: 50%; background: #888; flex-shrink:0; }}

    .footnote {{
      font-style: italic; font-size: 11px; color: #aaa; margin-top: 12px;
    }}

    /* Side boxes */
    .side-col {{
      flex: 1 1 0;
      display: flex;
      flex-direction: column;
      gap: 14px;
      padding-top: {side_padding_top}px;
    }}
    .info-box {{ border-radius: 10px; padding: 13px 15px 14px; }}
    .info-box.red    {{ background: #fdecea; border: 1.5px solid #f2aba0; }}
    .info-box.orange {{ background: #fef0ea; border: 1.5px solid #f5c4a0; }}
    .info-box.teal   {{ background: #e6f5f2; border: 1.5px solid #9ed8d1; }}
    .box-header {{ display: flex; align-items: center; gap: 8px; margin-bottom: 7px; }}
    .box-icon {{
      width: 28px; height: 28px; border-radius: 50%;
      display: flex; align-items: center; justify-content: center;
      font-size: 15px; font-weight: bold; flex-shrink: 0;
    }}
    .box-icon.red    {{ background: #C0392B; color: white; }}
    .box-icon.orange {{ background: #E05C3A; color: white; }}
    .box-icon.teal   {{ background: #1A7A6E; color: white; }}
    .box-title        {{ font-weight: bold; font-size: 13.5px; line-height: 1.3; }}
    .box-title.red    {{ color: #C0392B; }}
    .box-title.orange {{ color: #E05C3A; }}
    .box-title.teal   {{ color: #1A7A6E; }}
    .box-body {{ font-size: 12.5px; color: #444; line-height: 1.6; }}
    .hl.red    {{ color: #C0392B; font-weight: bold; }}
    .hl.orange {{ color: #E05C3A; font-weight: bold; }}
    .hl.teal   {{ color: #1A7A6E; font-weight: bold; }}

    .side-footnote {{
      font-size: 11px; color: #999; line-height: 1.5;
      margin-top: 6px; padding: 0 4px;
    }}
    .side-footnote .icon {{ font-size: 12px; }}
  </style>
</head>
<body>
<div class="outer">
  <div class="chart-col">

    <div class="info-pill">
      <span class="icon">&#9432;</span>
      Lower scores indicate more severe issues. Score 1 is best, 4 is worst.
    </div>

    <div id="plotly-chart"></div>

    <div class="legend-row">
      <div class="legend-item" style="left:{dot_legend_left:.0f}px;">
        <span class="dot-swatch"></span>
        Each dot = one inspection
      </div>
      <div class="legend-item" style="left:{line_legend_left:.0f}px;">
        <span class="line-swatch">
          <span class="seg"></span>
          <span class="circle"></span>
          <span class="seg"></span>
        </span>
        White line = median
      </div>
    </div>

    <div class="footnote">
      Note: Scores range from 1 (best) to 4 (worst). Violins show the distribution of all inspection outcomes within each category group.
    </div>
  </div>

  <div class="side-col">
    <div class="info-box red">
      <div class="box-header">
        <div class="box-icon red">&#8599;</div>
        <span class="box-title red">Higher median<br>(worse outcomes)</span>
      </div>
      <div class="box-body">
        Retail &amp; Shops has the highest median score
        (<span class="hl red">2.10</span>), meaning inspections in this
        category are typically less favorable.
      </div>
    </div>

    <div class="info-box orange">
      <div class="box-header">
        <div class="box-icon orange">&#9651;</div>
        <span class="box-title orange">Wide spread</span>
      </div>
      <div class="box-body">
        Retail &amp; Shops and Specialized / High-risk Regulated show wider
        distributions, indicating more variability in inspection results.
      </div>
    </div>

    <div class="info-box teal">
      <div class="box-header">
        <div class="box-icon teal">&#10003;</div>
        <span class="box-title teal">Lower median<br>(better outcomes)</span>
      </div>
      <div class="box-body">
        Wholesale &amp; Food Contact Materials has the lowest median score
        (<span class="hl teal">1.30</span>), indicating more favorable
        inspection outcomes overall.
      </div>
    </div>

    <div class="side-footnote">
      <span class="icon">&#9432;</span>
      Risk is based on the share of inspections with kontrol score 3 or 4.
      Based on 165,102 inspections.
    </div>
  </div>
</div>

<script>
  var figData = {plotly_json};
  Plotly.newPlot('plotly-chart', figData.data, figData.layout, {{
    responsive: false,
    displayModeBar: true,
    modeBarButtonsToRemove: ['lasso2d', 'select2d'],
    toImageButtonOptions: {{ format: 'png', filename: 'inspection_outcomes_chart', scale: 2 }}
  }});
</script>
</body>
</html>"""

with open("../figures/violin_chart.html", "w", encoding="utf-8") as f:
    f.write(html)

print("Saved violin_chart.html")
print(f"Fig size: {FIG_W}x{FIG_H}px  |  Plot area: {PLOT_W}px wide")

Saved violin_chart.html
Fig size: 980x680px  |  Plot area: 680px wide


# 2 - Drill down

In [ ]:
import json
import pandas as pd
import plotly.graph_objects as go

# ── 1. DATA ───────────────────────────────────────────────────────────────────
filtered = df[df["category_group"] != "Preliminary Registrations"].copy()
filtered["is_severe"] = filtered["kontrol"].isin([3, 4])
city_avg = filtered["is_severe"].mean()
group_stats = (
    filtered.groupby("category_group")["is_severe"]
    .mean().reset_index()
    .rename(columns={"is_severe": "severe_rate"})
)
group_stats["relative_risk"] = (group_stats["severe_rate"] / city_avg).round(2)
risk_df = group_stats[["category_group", "relative_risk"]].sort_values(
    "relative_risk", ascending=True
).reset_index(drop=True)

# ── SUBGROUP DATA FOR DRILLDOWN ───────────────────────────────────────────────
subgroup_stats = (
    filtered.groupby(["category_group", "category"])["is_severe"]
    .agg(severe_rate="mean", inspections="size")
    .reset_index()
)

subgroup_stats["relative_risk"] = (
    subgroup_stats["severe_rate"] / city_avg
).round(2)

subgroup_stats = subgroup_stats[subgroup_stats["inspections"] >= 100]

details = {}

for group, sub in subgroup_stats.groupby("category_group"):
    sub = sub.sort_values("relative_risk", ascending=True)

    details[group] = {
        "y": sub["category"].tolist(),
        "x": sub["relative_risk"].astype(float).tolist(),
        "colors": [pick_color(r) for r in sub["relative_risk"]],
        "hover": [
            f"<b>{row['category']}</b><br>"
            f"Relative risk: <b>{row['relative_risk']:.2f}x</b><br>"
            f"Inspections: {row['inspections']:,}"
            for _, row in sub.iterrows()
        ]
    }

# ── 2. COLORS ─────────────────────────────────────────────────────────────────
ABOVE_AVG_DARK  = "#C0392B"
ABOVE_AVG_LIGHT = "#E05C3A"
BELOW_AVG_DARK  = "#1A7A6E"
BELOW_AVG_LIGHT = "#2AB09A"

def pick_color(rr):
    if rr >= 1.30:   return ABOVE_AVG_DARK
    elif rr >= 1.0:  return ABOVE_AVG_LIGHT
    elif rr <= 0.40: return BELOW_AVG_DARK
    else:            return BELOW_AVG_LIGHT

bar_colors = [pick_color(r) for r in risk_df["relative_risk"]]

def hover_text(row):
    rr, cat = row["relative_risk"], row["category_group"]
    diff = abs(round((rr - 1.0) * 100))
    if rr > 1.0:   msg = f"<b>{diff}% more likely</b> than average to receive a severe outcome"
    elif rr < 1.0: msg = f"<b>{diff}% less likely</b> than average to receive a severe outcome"
    else:          msg = "exactly at the city average"
    return f"<b>{cat}</b><br>Relative risk: <b>{rr:.2f}x</b><br>{msg}"

hover_texts = [hover_text(row) for _, row in risk_df.iterrows()]

# ── 3. FIGURE ─────────────────────────────────────────────────────────────────
# left margin = space for y-axis labels (longest label ~32 chars at ~7px/char)
L_MARGIN = 240
PLOT_W   = 700   # actual plotting area width in px
FIG_W    = L_MARGIN + PLOT_W + 20  # total figure width

x_values = [float(r) for r in risk_df["relative_risk"]]

fig = go.Figure()

fig.add_trace(go.Bar(
    y=list(risk_df["category_group"]),
    x=x_values,
    orientation="h",
    marker=dict(color=bar_colors, line=dict(width=0)),
    text=[f"  {r:.2f}x" for r in x_values],
    textposition="outside",
    textfont=dict(size=13, color=bar_colors, family="Georgia, serif"),
    hovertemplate="%{customdata}<extra></extra>",
    customdata=hover_texts,
    cliponaxis=False,
    constraintext="none",
))

fig.add_vline(
    x=1.0,
    line=dict(color="#999999", dash="dash", width=1.5),
    annotation_text="City average",
    annotation_position="top",
    annotation_font=dict(size=11, color="#777777", family="Georgia, serif"),
)

fig.update_layout(
    title=dict(
        text=(
            "<b>Inspection Risk Depends More on What You Do<br>Than Where You Are</b>"
            "<br><sup>Relative risk of severe inspection outcomes by business category "
            "(compared to citywide average)</sup>"
        ),
        font=dict(size=21, family="Georgia, serif", color="#1a1a1a"),
        x=0, xanchor="left",
        pad=dict(l=0, t=0),
    ),
    xaxis=dict(
        tickfont=dict(size=12, color="#555", family="Georgia, serif"),
        range=[0, 2.35],   # wider so outside labels (1.43x) don't get clipped
        dtick=0.2,
        gridcolor="#e0e0e0",
        zeroline=True, zerolinecolor="#bbb", zerolinewidth=1,
        showline=False,
        fixedrange=True,
    ),
    yaxis=dict(
        tickfont=dict(size=13, color="#222", family="Georgia, serif"),
        fixedrange=True,
        automargin=False,   # we control margin manually
    ),
    plot_bgcolor="#f7f6f2",
    paper_bgcolor="#f7f6f2",
    margin=dict(l=L_MARGIN, r=60, t=155, b=30),  # t=155 gives title+subtitle room; r=60 for label breathing room
    height=480,  # taller to absorb extra top margin without squishing bars
    width=FIG_W,
    autosize=False,
    hoverlabel=dict(bgcolor="white", bordercolor="#ccc",
                    font=dict(size=13, family="Georgia, serif")),
    showlegend=False,
    bargap=0.38,
)

class _SafeEncoder(json.JSONEncoder):
    def default(self, obj):
        import numpy as np
        if isinstance(obj, (np.integer,)):  return int(obj)
        if isinstance(obj, (np.floating,)): return float(obj)
        if isinstance(obj, np.ndarray):     return obj.tolist()
        return super().default(obj)

details_json = json.dumps(details, cls=_SafeEncoder)

plotly_json = json.dumps(fig.to_dict(), cls=_SafeEncoder)

# ── 4. Legend positions as % of PLOT_W ───────────────────────────────────────
# x-axis goes 0→2.05 over PLOT_W pixels
# "Lower risk" label at x=0.4 → 0.4/2.05 * PLOT_W + L_MARGIN
# "Average risk" at x=1.0, "Higher risk" at x=1.6
def x_to_px(xval):
    return L_MARGIN + (xval / 2.35) * PLOT_W

low_px  = x_to_px(0.4)
avg_px  = x_to_px(1.0)
high_px = x_to_px(1.6)

# ── 5. HTML ───────────────────────────────────────────────────────────────────
html = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>Inspection Risk Chart</title>
  <script src="https://cdn.plot.ly/plotly-2.27.0.min.js"></script>
  <style>
    * {{ box-sizing: border-box; margin: 0; padding: 0; }}

    body {{
      background: #f7f6f2;
      font-family: Georgia, serif;
      padding: 32px 28px 48px;
    }}

    .outer {{
      display: flex;
      align-items: flex-start;
      gap: 24px;
      max-width: 1200px;
      margin: 0 auto;
    }}

    /* Chart column — exactly as wide as the Plotly figure */
    .chart-col {{
      width: {FIG_W}px;
      flex-shrink: 0;
      position: relative;
    }}

    #plotly-chart {{ width: {FIG_W}px; display: block; }}

    /* Legend sits below chart, labels pixel-positioned to match axis ticks */
    .legend-row {{
      position: relative;
      height: 38px;
      width: {FIG_W}px;
    }}

    .legend-item {{
      position: absolute;
      transform: translateX(-50%);
      text-align: center;
      white-space: nowrap;
    }}

    .legend-label {{ font-size: 13px; font-weight: bold; }}
    .legend-label.teal {{ color: #2AB09A; }}
    .legend-label.grey {{ color: #888; }}
    .legend-label.red  {{ color: #C0392B; }}
    .legend-sub {{ font-size: 11px; color: #999; margin-top: 1px; }}

    .xaxis-title {{
      text-align: center;
      margin-left: {L_MARGIN}px;
      width: {PLOT_W}px;
      font-size: 12px;
      color: #555;
      margin-top: 4px;
    }}

    .footnote {{
      font-style: italic;
      font-size: 11px;
      color: #aaa;
      margin-top: 10px;
    }}

    /* Side annotation boxes */
    .side-col {{
      flex: 1 1 0;
      display: flex;
      flex-direction: column;
      gap: 14px;
      padding-top: 100px;
    }}

    .info-box {{ border-radius: 10px; padding: 13px 15px 14px; }}
    .info-box.red    {{ background: #fdecea; border: 1.5px solid #f2aba0; }}
    .info-box.orange {{ background: #fef0ea; border: 1.5px solid #f5c4a0; }}
    .info-box.teal   {{ background: #e6f5f2; border: 1.5px solid #9ed8d1; }}

    .box-header {{ display: flex; align-items: center; gap: 8px; margin-bottom: 7px; }}

    .box-icon {{
      width: 28px; height: 28px; border-radius: 50%;
      display: flex; align-items: center; justify-content: center;
      font-size: 15px; font-weight: bold; flex-shrink: 0;
    }}
    .box-icon.red    {{ background: #C0392B; color: white; }}
    .box-icon.orange {{ background: #E05C3A; color: white; }}
    .box-icon.teal   {{ background: #1A7A6E; color: white; }}

    .box-title        {{ font-weight: bold; font-size: 13.5px; }}
    .box-title.red    {{ color: #C0392B; }}
    .box-title.orange {{ color: #E05C3A; }}
    .box-title.teal   {{ color: #1A7A6E; }}

    .box-body {{ font-size: 12.5px; color: #444; line-height: 1.6; }}
    .hl.red    {{ color: #C0392B; font-weight: bold; }}
    .hl.orange {{ color: #E05C3A; font-weight: bold; }}
    .hl.teal   {{ color: #1A7A6E; font-weight: bold; }}
  </style>
</head>
<body>
<div class="outer">
  <div class="chart-col">
    <button id="back-button" style="
    display:none;
    margin-bottom:10px;
    padding:8px 12px;
    border-radius:8px;
    border:1px solid #ccc;
    background:white;
    font-family:Georgia, serif;
    cursor:pointer;
    ">
    ← Back to category groups
    </button>
    <div id="plotly-chart"></div>

    <!-- Legend labels pixel-aligned to x-axis positions -->
    <div class="legend-row">
      <div class="legend-item" style="left:{low_px:.0f}px;">
        <div class="legend-label teal">Lower risk</div>
        <div class="legend-sub">(below average)</div>
      </div>
      <div class="legend-item" style="left:{avg_px:.0f}px;">
        <div class="legend-label grey">Average risk</div>
        <div class="legend-sub">(equal to average)</div>
      </div>
      <div class="legend-item" style="left:{high_px:.0f}px;">
        <div class="legend-label red">Higher risk</div>
        <div class="legend-sub">(above average)</div>
      </div>
    </div>

    <div class="xaxis-title">Relative risk compared with average</div>
    <div class="footnote">
      Risk = share of inspections with kontrol 3 or 4 (Bad or Highest Risk). Based on 165,102 inspections.
    </div>
  </div>

  <div class="side-col">
    <div class="info-box red">
      <div class="box-header">
        <div class="box-icon red">&#8599;</div>
        <span class="box-title red">Highest risk</span>
      </div>
      <div class="box-body">
        Retail &amp; Shops are <span class="hl red">1.43&times;</span> more likely
        than the average business to receive a severe inspection outcome.
      </div>
    </div>

    <div class="info-box orange">
      <div class="box-header">
        <div class="box-icon orange">&#9651;</div>
        <span class="box-title orange">Above average</span>
      </div>
      <div class="box-body">
        Specialized / High-risk Regulated businesses are
        <span class="hl orange">1.21&times;</span> more likely than the average.
      </div>
    </div>

    <div class="info-box teal">
      <div class="box-header">
        <div class="box-icon teal">&#8600;</div>
        <span class="box-title teal">Lowest risk</span>
      </div>
      <div class="box-body">
        Wholesale &amp; Food Contact Materials businesses are
        <span class="hl teal">0.29&times;</span> less likely than the average.
      </div>
    </div>
  </div>

</div>

<script>
  var figData = {plotly_json};
  var details = {details_json};

  var overviewData = JSON.parse(JSON.stringify(figData.data));
  var overviewLayout = JSON.parse(JSON.stringify(figData.layout));

  Plotly.newPlot('plotly-chart', figData.data, figData.layout, {{
    responsive: false,
    displayModeBar: true,
    modeBarButtonsToRemove: ['lasso2d', 'select2d'],
    toImageButtonOptions: {{ format: 'png', filename: 'inspection_risk_chart', scale: 2 }}
  }});

  document.getElementById('plotly-chart').on('plotly_click', function(eventData) {{
    var selectedGroup = eventData.points[0].y;

    if (!details[selectedGroup]) return;

    var sub = details[selectedGroup];

    var detailTrace = {{
      y: sub.y,
      x: sub.x,
      type: 'bar',
      orientation: 'h',
      marker: {{ color: sub.colors, line: {{ width: 0 }} }},
      text: sub.x.map(v => "  " + v.toFixed(2) + "x"),
      textposition: "outside",
      textfont: {{
        size: 13,
        color: sub.colors,
        family: "Georgia, serif"
      }},
      hovertemplate: "%{{customdata}}<extra></extra>",
      customdata: sub.hover,
      cliponaxis: false,
      constraintext: "none"
    }};

    var maxX = Math.max(...sub.x);

    var detailLayout = JSON.parse(JSON.stringify(overviewLayout));
    detailLayout.title.text =
      "<b>Which businesses drive the risk?</b><br>" +
      "<sup>Detailed categories inside " + selectedGroup + "</sup>";

    detailLayout.xaxis.range = [0, Math.max(2.35, maxX + 0.45)];
    detailLayout.height = Math.max(480, sub.y.length * 34 + 190);

    Plotly.react('plotly-chart', [detailTrace], detailLayout);

    document.getElementById('back-button').style.display = 'inline-block';
  }});

  document.getElementById('back-button').onclick = function() {{
    Plotly.react('plotly-chart', overviewData, overviewLayout);
    document.getElementById('back-button').style.display = 'none';
  }};
</script>
</body>
</html>"""

with open("../figures/inspection_risk_chart.html", "w", encoding="utf-8") as f:
    f.write(html)

print(f"Saved. Chart area: {PLOT_W}px wide, left margin: {L_MARGIN}px")
print(f"Legend positions — Low: {low_px:.0f}px  Avg: {avg_px:.0f}px  High: {high_px:.0f}px")

Saved. Chart area: 700px wide, left margin: 240px
Legend positions — Low: 359px  Avg: 538px  High: 717px


# 3 - Geography comparison